# Randomness You Can Trust (QRNG)

*Warm-up · UTC Problem 5*

Build a quantum random number generator, run it on IonQ's simulator through qBraid, and compare it to software randomness.

**Your job:** build the generator, collect bits from the local simulator and IonQ's free simulator, and compare them with software randomness. A real-QPU run (Part 2) is optional and uses credits.


In [ ]:
# Use the qBraid SDK environment's kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid",) if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(f"This kernel has no {', '.join(missing)}. Pick the qBraid SDK environment's "
                      "kernel (Kernel > Change Kernel), then run this cell again.")

# qBraid Lab: install Qiskit Aer if this environment doesn't have it yet (one time, ~20 s)
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()

n_qubits = 8
shots = 256          # total bits = n_qubits * shots


In [ ]:
def qrng_circuit(n):
    """A circuit whose measurements are random bits."""
    qc = QuantumCircuit(n, n)
    # TODO: put every qubit in superposition, then measure.
    # Hint: qc.h(range(n)); qc.measure(range(n), range(n))
    raise NotImplementedError("Build the QRNG circuit")
    return qc


In [ ]:
def bits_from_counts(counts):
    # Counts say how often each bitstring came up, not in what order, so this puts
    # identical results side by side. Fine for counting 1s, not for patterns.
    out = ""
    for bitstring, c in counts.items():
        out += bitstring * c
    return out

qc = qrng_circuit(n_qubits)
# memory=True keeps every shot in the order it ran, so pattern tests are fair
shots_in_order = sim.run(transpile(qc, sim), shots=shots, memory=True).result().get_memory()
sim_bits = "".join(shots_in_order)
print("Simulator bits collected:", len(sim_bits))


### Run it on a quantum computer through qBraid

Practice first on IonQ's simulator. It's free, and its `noise_model` option adds IonQ's gate noise. That noise sits mostly on two-qubit gates, so a circuit with few of them (like this one) comes back almost noise-free; real hardware adds readout error on top. When your circuit works, switch the two lines to the real IonQ QPU for your hardware run. That run uses credits, so cap it at 200 shots and copy the job ID into your write-up.

In [ ]:
# --- Run on a quantum computer through qBraid ---
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()   # in qBraid Lab this uses your account automatically

# 1) Practice for free: IonQ's simulator with a realistic noise model
device = provider.get_device("ionq:ionq:sim:simulator")
job = device.run(qc, shots=200, runtime_options={"noise_model": "aria-1"})

# 2) Your real hardware run (uses credits). Swap these in when you're ready:
# device = provider.get_device("ionq:ionq:qpu:forte-1")
# job = device.run(qc, shots=200)

print("Job ID:", job.id)          # copy this into your write-up
hw_counts = job.result().data.get_counts()
print(hw_counts)
hardware_bits = bits_from_counts(hw_counts)
print("Hardware bits collected:", len(hardware_bits))

# Heads up: the free simulator is still software, so its bits are pseudo-random, not truly random.
# Results come back as counts sorted by bitstring, not in shot order, so hardware_bits looks like
# a count-up pattern. Use them for tests that count (like monobit), not for tests that look at
# order (like longest_run). Part 2 (optional) asks for bits from a real QPU (option 2 above).

In [ ]:
# Software pseudo-random bits (provided)
rng = np.random.default_rng()
sw_bits = "".join(str(b) for b in rng.integers(0, 2, len(sim_bits)))


In [ ]:
def monobit(bits):
    """Fraction of 1s — should be near 0.5 for good randomness."""
    return bits.count("1") / len(bits)

def longest_run(bits):
    """Longest run of identical bits — a very simple pattern check."""
    best = run = 1
    for i in range(1, len(bits)):
        run = run + 1 if bits[i] == bits[i-1] else 1
        best = max(best, run)
    return best

for name, bits in [("simulator", sim_bits), ("software", sw_bits)]:
    print(f"{name:10s}  monobit={monobit(bits):.3f}  longest_run={longest_run(bits)}")

# TODO: add your hardware_bits to the comparison, and add one more test of your choice.
# hardware_bits are grouped by result, so only count-based tests are fair for them:
# monobit, or how often each qubit reads 1. longest_run needs shot order.


## What to submit
- **Part 1:** the working QRNG and a sample of bits.
- **Part 2 (optional, uses credits):** a bitstream from a real QPU.
- **Part 3:** the three-way test table (simulator, hardware, software).
- **Part 4:** the written explanation of why only hardware is truly unpredictable.
